# Topic 1: Delta Lake — ACID Transactions & Time Travel

> **Phase 5 → Week 9 → Topic 1**
>
> Prerequisites: Week 8 (Reading/Writing, Incremental Patterns)

---

## What is Delta Lake?

Delta Lake is an open-source storage layer on top of Parquet that adds:

```
Plain Parquet:                    Delta Lake:
  /data/table/                      /data/table/
    part-000.parquet                  part-000.parquet
    part-001.parquet                  part-001.parquet
    part-002.parquet          +       _delta_log/
                                        00000000000000000000.json  ← transaction log
                                        00000000000000000001.json
                                        00000000000000000002.json
                                        00000000000000000010.checkpoint.parquet

The _delta_log is what makes Delta Lake special:
  ✓ ACID transactions    (atomic writes — all or nothing)
  ✓ Time travel          (query any previous version)
  ✓ Audit log            (who wrote what, when)
  ✓ Schema enforcement   (reject writes that don't match the schema)
  ✓ Schema evolution     (add columns safely with ALTER TABLE)
  ✓ MERGE (upsert)       (UPDATE + INSERT in one atomic operation)
  ✓ OPTIMIZE + Z-ORDER   (compact files, co-locate related data)
```

---

## Interview Cheat Sheet

**Q: What is the Delta Lake transaction log?**
> The `_delta_log` directory contains a JSON file for every transaction. Each log entry records which files were added/removed. Spark constructs the current state of the table by replaying the log from the latest checkpoint. This is what enables time travel (replay only N entries), ACID (a transaction is one atomic log entry), and concurrent writes (optimistic concurrency control via log conflicts).

**Q: What is Delta Lake time travel?**
> Time travel lets you query a Delta table at a previous version or timestamp. `spark.read.format('delta').option('versionAsOf', 3).load('/path')` reads the table as it was at version 3. `option('timestampAsOf', '2024-01-15')` reads at a specific timestamp. Useful for: auditing, debugging wrong writes, reproducing ML training data, and rolling back bad updates.

**Q: What is VACUUM in Delta Lake?**
> `VACUUM` deletes old Parquet files that are no longer referenced by any Delta log version within the retention period (default 7 days). Without VACUUM, old files accumulate indefinitely. After VACUUM, you can no longer time-travel to versions older than the retention threshold. Always run VACUUM regularly in production to control storage costs.

**Q: What is OPTIMIZE and Z-ORDER in Delta Lake?**
> `OPTIMIZE` compacts many small Parquet files into fewer large files (target ~1GB each), improving read performance. `ZORDER BY (col)` co-locates related data in the same files by multi-dimensional clustering, so queries filtering on that column read fewer files. OPTIMIZE + ZORDER together = fewer files, each containing co-located data.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Week9 - Delta Lake") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

DELTA_PATH = "/tmp/delta_orders"
print("Delta Lake ready. Table path:", DELTA_PATH)

## Part 1: Creating a Delta Table

In [ ]:
import shutil, os
if os.path.exists(DELTA_PATH): shutil.rmtree(DELTA_PATH)

# Version 0: initial data
orders_v0 = spark.createDataFrame([
    ("O001", "C001", 250.0, "pending"),
    ("O002", "C002",  89.5, "pending"),
    ("O003", "C001", 500.0, "pending"),
], ["order_id", "customer_id", "amount", "status"])

orders_v0.write.format("delta").mode("overwrite").save(DELTA_PATH)
print("Version 0 written — initial 3 orders")

# Read it back
spark.read.format("delta").load(DELTA_PATH).show()

In [ ]:
# Version 1: append new orders
orders_v1 = spark.createDataFrame([
    ("O004", "C003",  45.0, "pending"),
    ("O005", "C002", 300.0, "pending"),
], ["order_id", "customer_id", "amount", "status"])

orders_v1.write.format("delta").mode("append").save(DELTA_PATH)
print("Version 1 — appended 2 orders")

# Version 2: overwrite all (simulate a full refresh)
orders_v2 = spark.createDataFrame([
    ("O001", "C001", 250.0, "delivered"),  # status changed
    ("O002", "C002",  89.5, "delivered"),
    ("O003", "C001", 500.0, "pending"),
    ("O004", "C003",  45.0, "cancelled"),  # status changed
    ("O005", "C002", 300.0, "pending"),
], ["order_id", "customer_id", "amount", "status"])

orders_v2.write.format("delta").mode("overwrite").save(DELTA_PATH)
print("Version 2 — overwrite with updated statuses")

print("\nCurrent state (version 2):")
spark.read.format("delta").load(DELTA_PATH).show()

## Part 2: Time Travel

In [ ]:
from delta.tables import DeltaTable

# Show the transaction history
dt = DeltaTable.forPath(spark, DELTA_PATH)
print("Delta transaction history:")
dt.history().select("version", "timestamp", "operation", "operationParameters").show(truncate=False)

# Time travel by version number
print("Version 0 (3 orders, all pending):")
spark.read.format("delta").option("versionAsOf", 0).load(DELTA_PATH).show()

print("Version 1 (5 orders, all pending):")
spark.read.format("delta").option("versionAsOf", 1).load(DELTA_PATH).show()

print("Version 2 (5 orders, some delivered/cancelled):")
spark.read.format("delta").load(DELTA_PATH).show()

## Part 3: MERGE (Upsert)

In [ ]:
# Incoming batch: updates to existing + new orders
incoming = spark.createDataFrame([
    ("O001", "C001", 250.0, "delivered"),   # already delivered — no change
    ("O003", "C001", 500.0, "delivered"),   # update: pending → delivered
    ("O006", "C004", 175.0, "pending"),     # new order
    ("O007", "C002",  60.0, "pending"),     # new order
], ["order_id", "customer_id", "amount", "status"])

# MERGE: match on order_id
dt.alias("target") \
  .merge(
      incoming.alias("source"),
      "source.order_id = target.order_id"
  ) \
  .whenMatchedUpdateAll() \
  .whenNotMatchedInsertAll() \
  .execute()

print("After MERGE (version 3):")
spark.read.format("delta").load(DELTA_PATH).orderBy("order_id").show()

print("Updated history:")
dt.history().select("version", "operation").show()

In [ ]:
# MERGE with conditional update (only update if status changed)
incoming2 = spark.createDataFrame([
    ("O002", "C002", 89.5, "returned"),   # update to returned
    ("O006", "C004", 175.0, "delivered"), # update to delivered
], ["order_id", "customer_id", "amount", "status"])

dt.alias("t") \
  .merge(
      incoming2.alias("s"),
      "s.order_id = t.order_id"
  ) \
  .whenMatchedUpdate(
      condition="s.status != t.status",   # only update if status actually changed
      set={"status": "s.status"}
  ) \
  .execute()

print("After conditional MERGE:")
spark.read.format("delta").load(DELTA_PATH).orderBy("order_id").show()

## Part 4: Schema Evolution

In [ ]:
# Schema enforcement: Delta rejects writes with wrong/extra columns by default
print("Schema enforcement demo:")
wrong_schema = spark.createDataFrame([
    ("O008", "C001", 99.0, "pending", "new_column_value"),
], ["order_id", "customer_id", "amount", "status", "new_col"])

try:
    wrong_schema.write.format("delta").mode("append").save(DELTA_PATH)
except Exception as e:
    print(f"Schema enforcement blocked write: {type(e).__name__}")
    print("(new_col not in target schema → rejected)")

# Schema evolution: add the new column with mergeSchema
print("\nSchema evolution with mergeSchema=True:")
wrong_schema.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save(DELTA_PATH)

result = spark.read.format("delta").load(DELTA_PATH)
print("Schema after evolution:")
result.printSchema()
print("(existing rows have null for new_col):")
result.orderBy("order_id").show()

## Part 5: OPTIMIZE and VACUUM

In [ ]:
print("""
OPTIMIZE — Compact Small Files:
════════════════════════════════════════════════════════════════
Problem: many small Parquet files = too many file open() calls,
         slow listing, poor read performance.

# SQL
spark.sql("OPTIMIZE delta.`/path/to/table`")

# Python API
dt.optimize().executeCompaction()

# With Z-ORDER (co-locate data for common filter columns)
dt.optimize().executeZOrderBy("customer_id", "order_date")

After OPTIMIZE:
  Before: 200 × 5MB files = 1GB (slow)
  After:  1 × 1GB file     = 1GB (fast reads)

Z-ORDER means: rows with the same customer_id are written to the same
Parquet file → a filter on customer_id only reads a few files instead of all.

Typical schedule: run OPTIMIZE daily or after large batch writes.

════════════════════════════════════════════════════════════════
VACUUM — Delete Old Files:
════════════════════════════════════════════════════════════════
VACUUM removes Parquet files no longer referenced in the Delta log
AND older than the retention period.

# Default: 7 days retention
spark.sql("VACUUM delta.`/path/to/table`")

# Custom retention (minimum 168 hours = 7 days enforced by default)
spark.sql("VACUUM delta.`/path/to/table` RETAIN 168 HOURS")

After VACUUM:
  Old Parquet files are deleted from disk.
  Time travel to versions older than retention period → AnalysisException.

Typical schedule: weekly VACUUM to control storage costs.
════════════════════════════════════════════════════════════════
""")

# Run OPTIMIZE on our demo table
try:
    dt.optimize().executeCompaction()
    print("OPTIMIZE completed")
    dt.history().select("version", "operation").show()
except Exception as e:
    print(f"OPTIMIZE: {e}")

## Part 6: Change Data Feed

In [ ]:
# Delta Change Data Feed: track row-level changes (INSERT/UPDATE/DELETE)
CDF_PATH = "/tmp/delta_cdf_demo"
if os.path.exists(CDF_PATH): shutil.rmtree(CDF_PATH)

# Create table with CDF enabled
initial = spark.createDataFrame([
    ("O001", "pending"), ("O002", "pending"),
], ["order_id", "status"])

initial.write.format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite").save(CDF_PATH)

# Make changes
cdf_table = DeltaTable.forPath(spark, CDF_PATH)
updates = spark.createDataFrame([("O001", "delivered")], ["order_id", "status"])
cdf_table.alias("t").merge(updates.alias("s"), "s.order_id = t.order_id") \
         .whenMatchedUpdateAll().execute()

# Read the change feed: see exactly what changed
changes = spark.read.format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", 1) \
    .load(CDF_PATH)

print("Change Data Feed — what changed since version 1:")
changes.select("order_id", "status", "_change_type", "_commit_version").show()
print("_change_type: 'update_preimage' = before, 'update_postimage' = after")

## Exercises

1. Create a Delta table from the orders CSV. Append 3 new orders. Then overwrite with different data. Use `.history()` to confirm 3 versions exist. Read version 0 and version 2 and compare.
2. Write a MERGE that: updates the `status` of existing orders, inserts new orders, and DELETEs orders with `status='cancelled'`.
3. Add a new column `region STRING` to an existing Delta table using `mergeSchema=True`. Verify old rows have `null` for `region`.
4. What happens if you `VACUUM` with retention 0 hours? Why does Delta block this by default?
5. Explain the difference between Delta's `overwrite` mode and a `MERGE`. When would each produce different results on a table with existing data?

In [ ]:
# Exercise 2: MERGE with delete
DEMO_PATH = "/tmp/delta_merge_delete"
if os.path.exists(DEMO_PATH): shutil.rmtree(DEMO_PATH)

base = spark.createDataFrame([
    ("O001", "pending"), ("O002", "cancelled"),
    ("O003", "pending"), ("O004", "cancelled"),
], ["order_id", "status"])
base.write.format("delta").save(DEMO_PATH)

incoming = spark.createDataFrame([
    ("O001", "delivered"),  # update
    ("O005", "pending"),    # new insert
], ["order_id", "status"])

tgt = DeltaTable.forPath(spark, DEMO_PATH)

tgt.alias("t").merge(incoming.alias("s"), "s.order_id = t.order_id") \
   .whenMatchedUpdateAll() \
   .whenNotMatchedInsertAll() \
   .whenNotMatchedBySourceDelete(condition="t.status = 'cancelled'") \
   .execute()

print("After MERGE with delete (cancelled orders removed):")
spark.read.format("delta").load(DEMO_PATH).orderBy("order_id").show()